# CM4IR baseline on Google Colab
Select **Runtime → Change runtime type → GPU**. Run cells from top to bottom. This notebook evaluates the author-provided LSUN Cat SR×4 images currently downloadable with `gdown`. Each run writes a JSON manifest containing image and checkpoint hashes, plus the exact command and metrics. See `REPRODUCE.md` for scope and limitations.


In [ ]:
!nvidia-smi
import torch
assert torch.cuda.is_available(), 'Select a GPU runtime before continuing'
print('PyTorch', torch.__version__)


In [ ]:
%cd /content
!git clone https://github.com/winanhle/CM4IR-reproduction.git
%cd /content/CM4IR-reproduction
!git rev-parse HEAD
!pip install -q einops lpips tqdm pillow pyyaml pandas scipy gdown
!python -m pip freeze > /content/cm4ir_pip_freeze.txt


In [ ]:
from pathlib import Path
import urllib.request
ckpt = Path('/content/CM4IR-reproduction/exp/logs/lsun_cat/cd_cat256_lpips.pt')
ckpt.parent.mkdir(parents=True, exist_ok=True)
if not ckpt.exists():
    urllib.request.urlretrieve('https://openaipublic.blob.core.windows.net/consistency/cd_cat256_lpips.pt', ckpt)
print('checkpoint bytes:', ckpt.stat().st_size)


The official LSUN Cat validation folder contains more files than `gdown --folder` lists in one batch. This cell downloads the available subset into the `ImageFolder` class directory. The runner records every image filename and SHA-256 hash; results from different manifests must not be compared as the same split.


In [ ]:
!mkdir -p /content/CM4IR-reproduction/exp/datasets/lsun_cat/cat
!gdown --folder --remaining-ok 'https://drive.google.com/drive/folders/1b69dtdPQYjuYnmADzkxSyhPNqjQb6tyN' -O /content/CM4IR-reproduction/exp/datasets/lsun_cat/cat
from pathlib import Path
images = sorted(Path('/content/CM4IR-reproduction/exp/datasets/lsun_cat/cat').glob('*.png'))
print('validation images:', len(images))
assert images, 'Author-provided LSUN Cat images are missing'


In [ ]:
%cd /content/CM4IR-reproduction
!python scripts/run_lsun_cat_sr4.py


In [ ]:
from pathlib import Path
from zipfile import ZipFile, ZIP_DEFLATED
from google.colab import files
root = Path('/content/CM4IR-reproduction')
name = f'CM4IR_lsun_cat_sr4_{len(images)}images'
archive = Path('/content') / f'{name}_artifacts.zip'
with ZipFile(archive, 'w', ZIP_DEFLATED) as bundle:
    for file in (root / 'exp/image_samples' / name).glob('*'):
        if file.is_file():
            bundle.write(file, file.relative_to(root))
    for file in (root / 'exp/repro_runs' / f'{name}.json',
                 root / 'exp/repro_runs' / f'{name}.log',
                 Path('/content/cm4ir_pip_freeze.txt')):
        bundle.write(file, file.name)
print('Artifact archive:', archive, archive.stat().st_size, 'bytes')
files.download(str(archive))
